# vector stores and retrievers
# we will cover
# documents, vector storage, and retrievers

In [ ]:
from langchain_core.documents import Document

# Create a comprehensive document collection for your vector database
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent animals, often appreciated for their playful and curious nature.",
        metadata={"source": "mammal-pets-doc1"},
    ),
    Document(
        page_content="Birds are known for their ability to fly and their diverse range of species.",
        metadata={"source": "mammal-pets-doc2"},
    ),
    Document(
        page_content="Rabbits are highly social herbivores that thrive when given plenty of open space to hop and fresh hay to chew.",
        metadata={"source": "mammal-pets-doc5"},
    ),
    Document(
        page_content="Hamsters are nocturnal rodents that enjoy running on exercise wheels, tunnel systems, and burrowing in deep bedding.",
        metadata={"source": "mammal-pets-doc4"},
    ),
    Document(
        page_content="Goldfish are popular aquatic pets that require clean, well-filtered water conditions and ample tank space to grow healthy and strong.",
        metadata={"source": "aquatic-pets-doc"},
    ),
    Document(
        page_content="Bearded dragons are docile reptiles that require specialized UVB lighting, a warm basking spot, and a balanced diet of insects and greens.",
        metadata={"source": "reptile-pets-doc"},
    ),
    Document(
        page_content="Chameleons are remarkable reptiles famous for their ability to change color and move their eyes independently.",
        metadata={"source": "reptile-pets-doc1"},
    ),
    Document(
        page_content="Canaries are small songbirds kept for their vibrant yellow color and beautiful melodic singing voices.",
        metadata={"source": "avian-pets-doc"},
    ),
    Document(
        page_content="Hermit crabs are unique invertebrates that do not grow their own shells, relying instead on abandoned snail shells for protection.",
        metadata={"source": "exotic-pets-doc"},
    ),
    Document(
        page_content="Guinea pigs are gentle, social rodents that communicate through a variety of vocalizations and enjoy living in pairs or small groups.",
        metadata={"source": "mammal-pets-doc6"},
    ),
    Document(
        page_content="Ferrets are curious and energetic mammals that require several hours of supervised playtime outside their cage each day.",
        metadata={"source": "mammal-pets-doc7"},
    ),
    Document(
        page_content="Parrots are highly intelligent birds capable of mimicking human speech and forming strong emotional bonds with their owners.",
        metadata={"source": "avian-pets-doc1"},
    ),
    Document(
        page_content="Turtles are long-lived reptiles that need both a dry basking area and clean water for swimming to maintain their health.",
        metadata={"source": "reptile-pets-doc2"},
    ),
    Document(
        page_content="Betta fish are territorial freshwater fish known for their vivid coloration and flowing fins, and should be housed separately from other bettas.",
        metadata={"source": "aquatic-pets-doc1"},
    ),
    Document(
        page_content="Geckos are small nocturnal lizards that use specialized toe pads to climb smooth vertical surfaces with ease.",
        metadata={"source": "reptile-pets-doc3"},
    ),
    Document(
        page_content="Hedgehogs are solitary insectivores covered in protective quills that curl into a tight ball when they feel threatened.",
        metadata={"source": "exotic-pets-doc1"},
    ),
    Document(
        page_content="Cockatiels are affectionate parrots native to Australia that enjoy whistling tunes and perching on their owner's shoulder.",
        metadata={"source": "avian-pets-doc2"},
    ),
    Document(
        page_content="Axolotls are aquatic salamanders that retain their larval features throughout their entire lives and can regenerate lost limbs.",
        metadata={"source": "aquatic-pets-doc2"},
    ),
    Document(
        page_content="Chinchillas have exceptionally dense and soft fur that requires regular dust baths to stay clean and free of moisture.",
        metadata={"source": "exotic-pets-doc2"},
    ),
]

documents


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")


from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
llm=ChatGroq(groq_api_key=groq_api_key,model="llama-3.1-8b-instant")

llm
             

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

embeddings

In [ ]:
# vector storage chroma
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embedding=embeddings)
vectorstore


In [ ]:
vectorstore.similarity_search("cat")


In [ ]:
# async querying
await vectorstore.asimilarity_search("cat")

In [ ]:
vectorstore.similarity_search_with_score("cat")


In [ ]:
# retrievers

from typing import List, Tuple

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

In [ ]:
# vector storage implement as a retriever
retriever = vectorstore.as_retriever(
    seardh_type="similarity",
    search_kwargs={"k": 1}
)
retriever.batch(["cat", "dog"])

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

groq_key = os.getenv("GROQ_API_KEY")
if groq_key is None:
    print("Error: GROQ_API_KEY not found in environment variables.")
elif not groq_key.startswith("gsk_"):
    print("Warning: your key doesn't start with 'gsk_'. Make sure you copied the right key.")
else:
    print("GROQ_API_KEY is set correctly from environment.")


In [ ]:
# RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Answer these questions using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response=rag_chain.invoke("Tell me about dogs.")
print(response.content)